In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json

import os
import numpy as np

import warnings
warnings.filterwarnings('ignore')

import MEArec as mr
import pandas as pd

import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle

In [4]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import scipy.signal
from scipy.spatial import ConvexHull
from typing import Dict, Iterable, List, Sequence, Tuple
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils import data
from torch.utils.data import random_split
from sklearn.metrics import accuracy_score, f1_score

import spikeinterface.extractors as se


# ==================== 0. Clique Building ====================

@dataclass
class CliqueInfo:
    """Information about a clique (subset of channels)"""
    clique_id: int
    device_channel_indices: List[int]
    contact_ids: List[str]
    center: Tuple[float, float]


def build_probe_group(probe_template_path: str = None):
    """
    Build probe group from MEArec template or return existing probe group.
    
    Parameters:
        probe_template_path: Path to MEArec recording file for probe template.
                            If None, uses default path.
    
    Returns:
        probegroup: ProbeGroup object with device_channel_indices set
    """
    if probe_template_path is None:
        probe_template_path = '/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_type1.h5'
    
    print("[INFO] Loading probe template")
    template_recording = se.MEArecRecordingExtractor(file_path=str(probe_template_path))
    probegroup = template_recording.get_probegroup()
    offset = 0
    for probe in probegroup.probes:
        n_contacts = probe.get_contact_count()
        device_indices = np.arange(offset, offset + n_contacts, dtype=int)
        probe.set_device_channel_indices(device_indices)
        offset += n_contacts
    return probegroup


def build_sliding_cliques(
    probe,
    clique_size: int = 50,
    min_size: int = 25,
    min_overlap: int = 16,
    target_groups: int = 11,
) -> List[CliqueInfo]:
    """
    Build sliding cliques from probe.
    
    Parameters:
        probe: Probe or ProbeGroup object
        clique_size: Size of each clique (number of channels), default 50
        min_size: Minimum size for a clique to be valid, default 25
        min_overlap: Minimum overlap between consecutive cliques, default 16
        target_groups: Target number of cliques to generate, default 11
    
    Returns:
        cliques: List of CliqueInfo objects
    """
    df = probe.to_dataframe()
    if "device_channel_indices" in df.columns:
        device_indices = df["device_channel_indices"].astype(int).to_numpy()
    else:
        device_indices = np.arange(len(df), dtype=int)
    positions = df.loc[:, ["x", "y"]].to_numpy()
    contact_ids = df["contact_ids"].astype(str).to_numpy()

    # Sort by y-coordinate (vertical position)
    order = np.argsort(positions[:, 1])
    ordered_device = device_indices[order]
    ordered_contacts = contact_ids[order]
    ordered_positions = positions[order]

    # Calculate step size for sliding window
    step = clique_size - min_overlap
    cliques: List[CliqueInfo] = []

    # Generate start indices for sliding windows
    start_indices = list(range(0, len(ordered_device) - clique_size + 1, step))
    if start_indices[-1] + clique_size < len(ordered_device):
        start_indices.append(len(ordered_device) - clique_size)

    # Build cliques
    for idx, start in enumerate(start_indices[:target_groups]):
        slice_device = ordered_device[start : start + clique_size]
        slice_positions = ordered_positions[start : start + clique_size]
        slice_contacts = ordered_contacts[start : start + clique_size]
        if len(slice_device) < min_size:
            continue
        center = tuple(np.mean(slice_positions, axis=0))
        cliques.append(
            CliqueInfo(
                clique_id=idx,
                device_channel_indices=list(slice_device),
                contact_ids=list(slice_contacts),
                center=center,
            )
        )

    print(f"[INFO] Built {len(cliques)} cliques (target {target_groups})")
    for info in cliques:
        print(
            f"       Clique {info.clique_id:02d}: channels {info.device_channel_indices[0]}-"
            f"{info.device_channel_indices[-1]} ({len(info.device_channel_indices)} channels)"
        )

    return cliques

In [2]:
recording, sorting = se.read_mearec("/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s.h5")
probe = recording.get_probe()


In [3]:
recording_recorded = spre.bandpass_filter(recording, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

In [7]:
output_folder = '/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s'
cliques = build_sliding_cliques(
    probe,
    clique_size=49,
    min_size=25,
    min_overlap=18,
    target_groups=12,
)

clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'clique_size': 49,
        'min_size': 25,
        'min_overlap': 18,
        'target_groups': 12,
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
}

clique_info_path = f'{output_folder}/clique_info.pkl'
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

[INFO] Built 12 cliques (target 12)
       Clique 00: channels 192-12 (49 channels)
       Clique 01: channels 103-115 (49 channels)
       Clique 02: channels 303-123 (49 channels)
       Clique 03: channels 23-227 (49 channels)
       Clique 04: channels 31-43 (49 channels)
       Clique 05: channels 326-338 (49 channels)
       Clique 06: channels 334-154 (49 channels)
       Clique 07: channels 54-258 (49 channels)
       Clique 08: channels 254-74 (49 channels)
       Clique 09: channels 165-177 (49 channels)
       Clique 10: channels 173-185 (49 channels)
       Clique 11: channels 371-383 (49 channels)


In [ ]:
# 处理合并数据的后处理：为每个clique生成neuron_inf和gt_detect_array，并拆分到各个日期
from tabnanny import verbose
import spikeinterface as si
import numpy as np
from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components
import pickle
import pandas as pd

mouse_name = 'mouse2'
combined_output_base = f'/media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/{mouse_name}'
dates_list = [1214, 1215, 1216, 1217, 1218, 1219]  # mouse1的日期列表，按顺序

# 构建cliques
probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
cliques = build_shank_cliques(probe, shank_boundaries=[250, 750, 1250])

print(f"\n{'='*60}")
print(f"读取并合并 {mouse_name} 的所有日期数据")
print(f"{'='*60}")

all_recordings_list = []
channel_list = None
# 记录每个日期的采样点数（在resample之前）
date_num_samples = {}  # {date: num_samples}

for date in dates_list:
    data_path = file_dict[mouse_name][date]
    
    # 获取该数据路径下的所有rhd文件
    file_list_path = Path(data_path)
    rhd_files = list(file_list_path.glob("*.rhd"))
    file_list = sorted(rhd_files)
    

    
    # 读取并合并该date的所有rhd文件
    recording_raw_list = []
    for file in file_list:
        recording_raw_list.append(se.read_intan(file, stream_id='0'))
    
    if len(recording_raw_list) > 0:
        date_recording = concatenate_recordings(recording_list=recording_raw_list)
        
        # 检测通道类型并选择对应的channel_list
        available_channels = date_recording.get_channel_ids()
        if 'A-127' in available_channels:
            channel_list = channel_list_A
        elif 'B-127' in available_channels:
            channel_list = channel_list_B
        else:
            print(f"警告: 未找到A-127或B-127通道，跳过此日期")
            continue
        
        # 选择通道
        date_recording = date_recording.select_channels(channel_list)
        
        # 统一将B开头的channel重命名为A开头
        channel_ids = date_recording.get_channel_ids()
        new_channel_ids = []
        renamed_count = 0
        for ch_id in channel_ids:
            if isinstance(ch_id, str) and ch_id.startswith('B-'):
                new_ch_id = 'A-' + ch_id[2:]
                new_channel_ids.append(new_ch_id)
                renamed_count += 1
            else:
                new_channel_ids.append(ch_id)
        
        if renamed_count > 0:
            date_recording = date_recording.rename_channels(new_channel_ids)
        
        # 记录该日期的原始采样点数（在resample之前）
        # 注意：这里记录的是原始采样点数，后续会resample到10000Hz
        # 但我们需要知道resample后的采样点数
        all_recordings_list.append(date_recording)


recording_combined = concatenate_recordings(recording_list=all_recordings_list)

print("\n开始预处理...")
recording_raw = spre.unsigned_to_signed(recording_combined)
recording_raw = spre.resample(recording_raw, 10000)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)

# 计算每个日期在resample后的采样点数
# 需要在resample之前记录原始采样点数和采样率，然后计算resample后的采样点数
target_sampling_frequency = 10000.0
for i, date in enumerate(dates_list):
    if i < len(all_recordings_list):
        date_recording = all_recordings_list[i]
        original_sampling_freq = date_recording.get_sampling_frequency()
        original_num_samples = date_recording.get_num_samples()
        # resample后的采样点数 = 原始采样点数 * (目标采样率 / 原始采样率)
        resampled_num_samples = int(original_num_samples * (target_sampling_frequency / original_sampling_freq))
        date_num_samples[date] = resampled_num_samples
        print(f"Date {date}: 原始采样点数 = {original_num_samples}, 原始采样率 = {original_sampling_freq:.1f} Hz, resample后采样点数 = {resampled_num_samples}")

# 计算每个segment的采样点范围（基于date_num_samples）
segment_sample_ranges_by_date = {}  # {segment_idx: (start_sample, end_sample)}
cumulative_samples = 0
for segment_idx, date in enumerate(dates_list):
    if date in date_num_samples:
        num_samples = date_num_samples[date]
        segment_sample_ranges_by_date[segment_idx] = (cumulative_samples, cumulative_samples + num_samples)
        cumulative_samples += num_samples
        print(f"Segment {segment_idx} (Date {date}): 采样点范围 = [{segment_sample_ranges_by_date[segment_idx][0]}, {segment_sample_ranges_by_date[segment_idx][1]})")

# 对每个clique进行处理
for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"处理 Clique {clique_id}")
    print(f"{'='*60}")
    
    clique_output_folder = f'{combined_output_base}/clique_{clique_id}'
    phy_folder = f'{clique_output_folder}/phy_folder_for_kilosort'
    
    # 读取sorting结果
    sorting_curated_phy = se.read_phy(phy_folder, exclude_cluster_groups=["noise"])
    print(f"读取到 {len(sorting_curated_phy.unit_ids)} 个units")
    
    recording_combined_clique = get_recording_clique(recording_f, clique)
    analyzer_curated_phy = si.create_sorting_analyzer(
        sorting=sorting_curated_phy, 
        recording=recording_combined_clique, 
        format='binary_folder',
        folder=clique_output_folder + '/analyzer_curated_temp',
        n_jobs=20, verbose = False
    )
    
    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "templates",
        "unit_locations",
        "template_similarity"
    ]
    
    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "template_similarity": {"method": "cosine_similarity"}
    }
    
    analyzer_curated_phy.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
    

    
    # 获取neuron信息（与concat_post中相同）
    templates_ext = analyzer_curated_phy.get_extension("templates")
    templates_dense = templates_ext.data["average"]
    sparsity = analyzer_curated_phy.sparsity
    unit_locations_ext = analyzer_curated_phy.get_extension("unit_locations")
    unit_locations = unit_locations_ext.get_data()
    channel_locations = analyzer_curated_phy.get_channel_locations()
    
    # 处理merge逻辑（与concat_post相同）
    if unit_locations.shape[1] >= 2:
        unit_distances = scipy.spatial.distance.cdist(
            unit_locations[:, :2], 
            unit_locations[:, :2], 
            metric="euclidean"
        )
    else:
        unit_distances = scipy.spatial.distance.cdist(
            unit_locations, 
            unit_locations, 
            metric="euclidean"
        )
    
    template_similarity_ext = analyzer_curated_phy.get_extension("template_similarity")
    template_similarity = template_similarity_ext.get_data()
    
    distance_threshold = 10.0
    similarity_threshold = 0.95
    num_units = len(analyzer_curated_phy.unit_ids)
    pair_mask = np.zeros((num_units, num_units), dtype=bool)
    
    for i in range(num_units):
        for j in range(i + 1, num_units):
            if unit_distances[i, j] < distance_threshold and template_similarity[i, j] > similarity_threshold:
                pair_mask[i, j] = True
                pair_mask[j, i] = True
    
    n_components, labels = connected_components(
        csgraph=pair_mask, 
        directed=False, 
        return_labels=True
    )
    
    merge_unit_groups = []
    unit_ids_list = analyzer_curated_phy.unit_ids
    for component_id in range(n_components):
        unit_indices = np.where(labels == component_id)[0]
        if len(unit_indices) > 1:
            group = [unit_ids_list[i] for i in unit_indices]
            merge_unit_groups.append(group)
    
    # 应用merge（如果有需要merge的units）
    if len(merge_unit_groups) > 0:
        analyzer_merged = analyzer_curated_phy.merge_units(
            merge_unit_groups=merge_unit_groups,
            censor_ms=0.3,
            merging_mode="hard",
            new_id_strategy="append",
            format='binary_folder',
            folder=clique_output_folder + '/analyzer_merged',
            verbose=True,
            n_jobs=20
        )
        
        analyzer_merged.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
        
        templates_ext_final = analyzer_merged.get_extension("templates")
        templates_dense_final = templates_ext_final.data["average"]
        sparsity_final = analyzer_merged.sparsity
        unit_locations_ext_final = analyzer_merged.get_extension("unit_locations")
        unit_locations_final = unit_locations_ext_final.get_data()
        channel_locations_final = analyzer_merged.get_channel_locations()
        sorting_final = analyzer_merged.sorting
        unit_ids_list_final = analyzer_merged.unit_ids
        
        # 生成position_waveforms
        position_waveforms_final = []
        for unit_id in unit_ids_list_final:
            unit_index = analyzer_merged.sorting.id_to_index(unit_id)
            template_dense_unit = templates_dense_final[unit_index, :, :]
            template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
            sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
            
            if len(sparse_channel_indices) == 0:
                position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
                position_waveforms_final.append(position_waveform)
                continue
            
            sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
            unit_location = unit_locations_final[unit_index, :2]
            
            distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
            epsilon = 1e-10
            weights = 1.0 / (distances + epsilon)
            weights = weights / np.sum(weights)
            
            position_waveform = np.dot(template_sparse_unit, weights)
            position_waveforms_final.append(position_waveform)
        
        position_waveforms_final = np.array(position_waveforms_final)
        extremum_channels_final = get_template_extremum_channel(
            analyzer_merged, 
            peak_sign="neg",
            outputs="id"
        )
    else:
        # 不需要merge，使用原始结果
        templates_ext_final = analyzer_curated_phy.get_extension("templates")
        templates_dense_final = templates_ext_final.data["average"]
        sparsity_final = analyzer_curated_phy.sparsity
        unit_locations_ext_final = analyzer_curated_phy.get_extension("unit_locations")
        unit_locations_final = unit_locations_ext_final.get_data()
        channel_locations_final = analyzer_curated_phy.get_channel_locations()
        sorting_final = analyzer_curated_phy.sorting
        unit_ids_list_final = unit_ids_list
        
        # 生成position_waveforms
        position_waveforms_final = []
        for unit_id in unit_ids_list_final:
            unit_index = analyzer_curated_phy.sorting.id_to_index(unit_id)
            template_dense_unit = templates_dense_final[unit_index, :, :]
            template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
            sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
            
            if len(sparse_channel_indices) == 0:
                position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
                position_waveforms_final.append(position_waveform)
                continue
            
            sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
            unit_location = unit_locations_final[unit_index, :2]
            
            distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
            epsilon = 1e-10
            weights = 1.0 / (distances + epsilon)
            weights = weights / np.sum(weights)
            
            position_waveform = np.dot(template_sparse_unit, weights)
            position_waveforms_final.append(position_waveform)
        
        position_waveforms_final = np.array(position_waveforms_final)
        extremum_channels_final = get_template_extremum_channel(
            analyzer_curated_phy, 
            peak_sign="neg",
            outputs="id"
        )
    
    # 获取channel_ids（用于将通道索引转换为通道名称）
    # 使用analyzer的recording来获取channel_ids
    if len(merge_unit_groups) > 0:
        channel_ids_list = list(analyzer_merged.recording.get_channel_ids())
    else:
        channel_ids_list = list(analyzer_curated_phy.recording.get_channel_ids())
    
    # 计算每个unit的channel_id（template中值不为0的通道）
    channel_ids_dict = {}  # {unit_id: [channel_id1, channel_id2, ...]}
    for idx, unit_id in enumerate(unit_ids_list_final):
        unit_index = sorting_final.id_to_index(unit_id)
        template_unit = templates_dense_final[unit_index, :, :]  # (n_samples, n_channels)
        
        # 找到template中值不为0的通道
        # 检查每个通道是否有非零值（在整个时间窗口内）
        non_zero_channels = []
        for ch_idx in range(template_unit.shape[1]):  # 遍历channels（最后一个维度）
            if np.any(template_unit[:, ch_idx] != 0):  # 检查该通道在所有时间点的值
                # 将通道索引转换为通道名称（格式和extremum_channel一样）
                channel_name = str(channel_ids_list[ch_idx])
                non_zero_channels.append(channel_name)
        
        channel_ids_dict[unit_id] = non_zero_channels
    
    # 计算channel_snr（每个unit的各个channel的SNR）
    print("计算channel_snr...")
    recording_combined_clique = get_recording_clique(recording_f, clique)
    n_channels = recording_combined_clique.get_num_channels()
    sampling_frequency = recording_combined_clique.get_sampling_frequency()
    
    # 1. 计算noise_std（使用和detect_spike相同的方法）
    # 读取一小段数据来计算noise_std（使用前10秒的数据）
    duration_samples = int(10 * sampling_frequency)  # 10秒
    max_samples = recording_combined_clique.get_num_samples()
    actual_samples = min(duration_samples, max_samples)
    traces = recording_combined_clique.get_traces(start_frame=0, end_frame=actual_samples)  # (n_timepoints, n_channels)

    noise_std_detect = np.median(np.abs(traces) / 0.6745, axis=0)  # (n_channels,)

    all_spike_times = []
    all_spike_unit_ids = []
    for unit_id in unit_ids_list_final:
        spike_train = sorting_final.get_unit_spike_train(unit_id)
        all_spike_times.extend(spike_train.tolist())
        all_spike_unit_ids.extend([unit_id] * len(spike_train))
    
    n_spikes_total = len(all_spike_times)
    n_spikes_sample = min(1000, n_spikes_total)
    if n_spikes_sample > 0:
        random_indices = np.random.choice(n_spikes_total, size=n_spikes_sample, replace=False)
        sampled_spike_times = [all_spike_times[i] for i in random_indices]
        sampled_spike_unit_ids = [all_spike_unit_ids[i] for i in random_indices]
    else:
        sampled_spike_times = []
        sampled_spike_unit_ids = []
    
    # 4. 提取这些spike的waveform并计算每个channel的负值amplitude
    left_sample = 10
    right_sample = 20
    window_size = left_sample + right_sample
    
    channel_snr_dict = {} 
    
    for unit_id in unit_ids_list_final:
        channel_snr_dict[unit_id] = {}
        unit_spike_times = [st for st, uid in zip(sampled_spike_times, sampled_spike_unit_ids) if uid == unit_id]
        
        if len(unit_spike_times) == 0:
            unit_spike_times = sorting_final.get_unit_spike_train(unit_id).tolist()
            if len(unit_spike_times) > 1000:
                unit_spike_times = np.random.choice(unit_spike_times, size=1000, replace=False).tolist()
        
        unit_waveforms = []  # List of (n_channels, window_size)
        valid_spike_times = []
        
        for spike_time in unit_spike_times:
            start = spike_time - left_sample
            end = spike_time + right_sample

            waveform = recording_combined_clique.get_traces(start_frame=start, end_frame=end)  # (n_channels, window_size)
            unit_waveforms.append(waveform)
            valid_spike_times.append(spike_time)
        
        if len(unit_waveforms) == 0:
            continue
        
        unit_waveforms = np.array(unit_waveforms)  # (n_spikes, n_channels, window_size)
        
        spike_time_values = unit_waveforms[:, left_sample, :]  # (n_spikes, n_channels) - 每个spike在spike_time时刻各个channel的值
        
        channel_amplitudes = np.mean(spike_time_values, axis=0)  # (n_channels,) - 每个channel的平均值（在spike_time时刻）
        channel_snr = np.abs(channel_amplitudes) / noise_std_detect  # (n_channels,)
        
        # 只保存 channel_ids_dict[unit_id] 中列出的通道的 SNR
        channel_ids_list = list(recording_combined_clique.get_channel_ids())
        unit_channel_ids = channel_ids_dict.get(unit_id, [])  # 获取该unit的channel_id列表
        
        for ch_idx, snr_value in enumerate(channel_snr):
            channel_id = str(channel_ids_list[ch_idx])
            # 只保存 channel_ids_dict 中列出的通道
            if channel_id in unit_channel_ids:
                channel_snr_dict[unit_id][channel_id] = float(snr_value)
    
    print(f"完成channel_snr计算，共处理{len(channel_snr_dict)}个units")
    
    # 生成整体的neuron_inf
    neuron_inf = {}
    for idx, unit_id in enumerate(unit_ids_list_final):
        neuron_inf[unit_id] = {
            'location_x': float(unit_locations_final[idx, 0]),
            'location_y': float(unit_locations_final[idx, 1]),
            'position_waveform': position_waveforms_final[idx],
            'extremum_channel': extremum_channels_final[unit_id],
            'channel_id': channel_ids_dict[unit_id],
            'channel_snr': channel_snr_dict.get(unit_id, {})  # 添加channel_snr字段
        }
    
    # 保存整体的neuron_inf
    with open(clique_output_folder + '/neuron_inf.pickle', 'wb') as f:
        pickle.dump(neuron_inf, f)
    
    # 生成整体的gt_detect_array（使用合并数据的时间）
    recording_combined_clique = get_recording_clique(recording_f, clique)
    sampling_frequency = recording_combined_clique.get_sampling_frequency()
    # 使用之前计算的segment_sample_ranges_by_date（基于日期划分）
    segment_sample_ranges = segment_sample_ranges_by_date.copy()
    
    spike_vector_final = sorting_final.to_spike_vector()
    gt_detect_data_all = []
    
    for spike in spike_vector_final:
        unit_index = spike['unit_index']
        unit_id = sorting_final.unit_ids[unit_index]
        sample_index = spike['sample_index']  # 全局采样点索引（跨所有日期）
        
        # 根据sample_index确定它属于哪个segment（日期）
        # sample_index是全局的，需要根据segment_sample_ranges来确定属于哪个segment
        segment_index = None
        for seg_idx, (start_sample, end_sample) in segment_sample_ranges.items():
            if start_sample <= sample_index < end_sample:
                segment_index = seg_idx
                break
        
        if segment_index is None:
            # 如果无法确定segment，跳过（理论上不应该发生）
            continue
        
        # 计算segment内的相对采样点索引（精确到每个采样点）
        segment_start_sample, segment_end_sample = segment_sample_ranges[segment_index]
        relative_sample_index = sample_index - segment_start_sample
        
        time_seconds = relative_sample_index
        
        extremum_channel = extremum_channels_final[unit_id]
        
        gt_detect_data_all.append({
            'time': time_seconds,
            'unit_id': unit_id,
            'extremum_channel': str(extremum_channel),
            'segment_index': segment_index  # 保存segment_index用于精确筛选
        })
    
    gt_detect_array_all = pd.DataFrame(gt_detect_data_all)
    
    gt_detect_array_all.to_csv(clique_output_folder + '/gt_detect_array.csv', index=False)
    

    for segment_idx, date in enumerate(dates_list):
        print(f"\n处理日期 {date} (Segment {segment_idx})...")
        
        # 使用segment_index进行精确筛选，而不是时间范围
        # 这样可以确保精确到每个采样点，避免浮点数精度问题
        mask = (gt_detect_array_all['segment_index'] == segment_idx)
        
        spikes_in_segment = gt_detect_array_all[mask].copy()
        
        # time已经是segment内的相对时间（从0开始），精确到采样点
        # 删除segment_index列（因为已经筛选完成，不再需要）
        if 'segment_index' in spikes_in_segment.columns:
            spikes_in_segment = spikes_in_segment.drop(columns=['segment_index'])
        
        # 筛选neuron：计算每个neuron的firing rate，如果 < 1 Hz则删除
        # 使用采样点数计算：firing_rate = spike_count * sampling_frequency / num_samples
        neurons_in_date = spikes_in_segment['unit_id'].unique()
        firing_rate_threshold = 0.5  # Hz
        valid_neurons = []
        removed_spikes_count = 0
        
        # 获取该segment的采样点数（用于计算firing rate）
        # 使用date_num_samples获取该日期在resample后的采样点数
        if date not in date_num_samples:
            print(f"  警告: 日期 {date} 的采样点数未找到，跳过")
            continue
        segment_num_samples = date_num_samples[date]
        
        for unit_id in neurons_in_date:
            if unit_id not in neuron_inf:
                # 如果neuron不在neuron_inf中，跳过
                continue
            
            # 计算该neuron在该segment中的spike数量
            neuron_spikes = spikes_in_segment[spikes_in_segment['unit_id'] == unit_id]
            spike_count = len(neuron_spikes)
            
            # 计算firing rate (spikes per second)，使用采样点数：spike_count * sampling_frequency / num_samples
            firing_rate = (spike_count * sampling_frequency) / segment_num_samples if segment_num_samples > 0 else 0
            
            if firing_rate >= firing_rate_threshold:
                valid_neurons.append(unit_id)
            else:
                # 删除该neuron的spikes
                removed_spikes_count += spike_count
                spikes_in_segment = spikes_in_segment[spikes_in_segment['unit_id'] != unit_id]
        
        # 只保留有效的neurons
        neuron_inf_date = {unit_id: neuron_inf[unit_id] for unit_id in valid_neurons}
        

        date_output_folder = f'{clique_output_folder}/date_{date}'
        os.makedirs(date_output_folder, exist_ok=True)
        
        with open(date_output_folder + '/neuron_inf.pickle', 'wb') as f:
            pickle.dump(neuron_inf_date, f)
        
        spikes_in_segment.to_csv(date_output_folder + '/gt_detect_array.csv', index=False)
        
        print(f"  日期 {date} 结果已保存到: {date_output_folder}")
        print(f"    - Neurons: {len(neuron_inf_date)}")
        print(f"    - Spikes: {len(spikes_in_segment)}")

print("\n所有clique处理完成！")
